In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
import importlib
from config.config import *

import os
from pyspark.sql import *

In [0]:
# base_path = "/Volumes/workspace/default/tourismdata"

# raw_path = f"{base_path}/raw"
# bronze_path = f"{base_path}/bronze_layer"
# silver_path = f"{base_path}/silver_layer"
# gold_path = f"{base_path}/gold_layer"

In [0]:
files = dbutils.fs.ls(raw_path)

for file in files:
    print(file.name)

def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

In [0]:
for file in files:
    filename = file.name
    file_path = file.path
    extension = filename.split(".")[-1].lower()

    if extension == "csv":
        df = (
            spark.read
            .option("header", False)
            .option("inferSchema", False)
            .csv(file_path)
        )

    elif extension == "tsv":
        df = (
            spark.read
            .option("header", False)
            .option("inferSchema", False)
            .option("delimiter", "\t")
            .csv(file_path)
        )

    else:
        print(f"Skipping unsupported file: {filename}")
        continue

    dataset_name = filename.rsplit(".", 1)[0].lower()
    delta_path = f"{bronze_path}/bronze_{dataset_name}"

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(delta_path)

    print(f"Bronze ingestion completed: {dataset_name}")